#### Visão Geral
##### Schema : gold
##### Table : dim_clientes

| Detalhe | Informação |
|---------|------------|
| Criado Originalmente Por | Wellikiandre Bosich |
| Tabela de Dados de Saída | `{environment}.gold.dim_clientes` |
| Origem Fonte de Dados de Entrada | Camada silver |
| Destino Fonte de Dados de Saída | Camada gold |

#### Histórico

| Data       | Desenvolvido Por         | Motivo                                         |
|:----------:|--------------------------|-----------------------------------------------|
| 04/06/2026 | Wellikiandre Bosich    | Criação da dimensão clientes com cruzamento e enriquecimento geográfico na Gold. |

In [ ]:
%run ../0_Config/0-Init

In [ ]:
# Parâmetros de Inicialização
table_name = 'dim_clientes'
output_path_data = f"{var_gold}/{table_name}/data"
table_name_schema = f'{var_environment}.{var_gold_schema}.{table_name}'

In [ ]:
from pyspark.sql.functions import col, sha2, coalesce, lit

df_clientes = spark.read.table(f"{var_environment}.{var_silver_schema}.case_crm_clientes")
df_regioes = spark.read.table(f"{var_environment}.{var_silver_schema}.case_legado_regioes")

df_gold = (
    df_clientes.join(df_regioes, df_clientes.uf_cliente == df_regioes.uf_estado, "left")
    .withColumn("sk_cliente", sha2(col("id_cliente").cast("string"), 256))
    .select(
        col("sk_cliente").cast("string").alias("sk_cliente"),
        col("id_cliente").cast("integer").alias("id_cliente"),
        coalesce(col("nome_cliente"), lit("Não Identificado")).cast("string").alias("nome_cliente"),
        coalesce(col("email_cliente"), lit("Sem E-mail")).cast("string").alias("email_cliente"),
        col("documento_cliente").cast("string").alias("documento_cliente"),
        col("tipo_documento_cliente").cast("string").alias("tipo_documento_cliente"),
        col("uf_cliente").cast("string").alias("uf_cliente"),
        coalesce(col("nome_regiao"), lit("Região Não Informada")).cast("string").alias("regiao_cliente"),
        col("data_cadastro_cliente").alias("data_cadastro")
    )
)

In [ ]:
process_data(
    df_write=df_gold,
    tipo_carga='delta',
    nome_gravacao_tabela=table_name_schema,
    caminho_gravacao_tabela=output_path_data,
    chave_clusterby=['regiao_cliente'],
    chave_upsert='sk_cliente'
)